## 5. (v2 파일럿은 완료 — 설계용, 확증서 제외)

이미 10쌍 P1a 파일럿으로 전파 교란을 발견해 v3 설계를 확정했다. 확증은 6번(홀드아웃).


## 0. GPU 확인
3B fp16은 T4(16GB)에 올라간다. GPU 없으면 런타임 유형을 T4로.


In [ ]:
!nvidia-smi -L


## 1. repo clone (main)
코드는 **main**에서. 이미 있으면 main 최신으로 정렬.


In [ ]:
REPO_URL = "https://github.com/deanjs/instruction-adherence.git"
BRANCH = "main"
import os
if not os.path.isdir("instruction-adherence"):
    !git clone --branch {BRANCH} {REPO_URL}
%cd instruction-adherence
!git checkout {BRANCH} && git pull origin {BRANCH}
!git log --oneline -1


## 2. 의존성
`AttentionInterface` 등록에 transformers>=4.51. Colab torch(CUDA)는 유지.


In [ ]:
!pip install -q "transformers>=4.51.0" "accelerate>=0.26.0"
import torch, transformers
print("transformers", transformers.__version__, "| cuda", torch.cuda.is_available())


## 3. 검증 (게이트) — 개입 하네스 불변식

**PASS여야 실측으로 넘어간다.** V1 표준경로=SDPA · V2 no-op 불변 · V3 지침 patch≈0 ·
V4 GQA 단위(P1a=KV group·P2=query head) · V5 P2 질량 보존 · V6 λ=1 항등 ·
V7 L25 이식이 점수를 움직이는지(sanity). 크기 무관 → 1.5B fp32.


In [ ]:
!python src/stage3_intervention.py --validate


## 4. A분할 — 기저 gap (준수 − 손상)

개입 없이 손상/준수 baseline만. gap = 완전 회복 목표 = Recovery Ratio 분모.
Drive에 append(재개 가능).


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
OUT = "/content/drive/MyDrive/instruction-adherence/stage3_intervention.jsonl"
os.makedirs(os.path.dirname(OUT), exist_ok=True)
!python src/stage3_intervention.py --run-a --n-seeds 3 --out "{OUT}"


## 5. (v2 파일럿은 완료 — 설계용, 확증서 제외)

이미 10쌍 P1a 파일럿으로 전파 교란을 발견해 v3 설계를 확정했다. 확증은 6번(홀드아웃).


In [ ]:
# (파일럿은 이미 수행됨 — 재실행 불필요. 필요시에만.)
# !python src/stage3_intervention.py --run-b --p1a --max-pairs 10 --n-seeds 1 --out "{OUT}"

## 6. 확증 (v3, 홀드아웃)

v2 파일럿(앞 10쌍)은 설계용 → **확증에서 제외**하고 `--pair-start 10` 홀드아웃에서만.
**주 검정(사전 등록)** = 대응 대비 two-way boot CI가 0 배제:
① 인과·내용 특이성(L25 준수 vs 무작위 donor) ② 층 특이성(L25 vs 후기 1/3 평균) ③ 회복>0.
전 층·후기 순위 p는 민감도로만(순위 p는 바닥 존재).


In [ ]:
# 확증(v3): v2 파일럿 10쌍 제외 → 홀드아웃(--pair-start 10). 별도 결과 파일.
OUT3 = "/content/drive/MyDrive/instruction-adherence/stage3_intervention_v3.jsonl"
import os; os.makedirs(os.path.dirname(OUT3), exist_ok=True)
!python src/stage3_intervention.py --run-a --n-seeds 3 --out "{OUT3}"
!python src/stage3_intervention.py --run-b --p1a --pair-start 10 --n-seeds 3 --out "{OUT3}"
!python src/stage3_intervention.py --run-b --p2 --pair-start 10 --n-seeds 3 --out "{OUT3}"

**(선택) 국소화 스윕** — group별·후보 층 query head 28개 순회.


In [ ]:
!python src/stage3_intervention.py --run-b --p1a --p2 --sweep --pair-start 10 --n-seeds 3 --out "{OUT3}"

## 7. 집계 — cluster bootstrap · 층 귀무 · Recovery Ratio · λ 추세

이름쌍·seed cluster bootstrap 95% CI, L25가 나머지 층(같은 규모) 귀무분포 상위 꼬리인지,
Recovery Ratio CI(분자·분모 독립 부트), P2 λ 단조 추세(Spearman).


In [ ]:
!python src/stage3_intervention.py --summary-only --out "{OUT3}"

## 8. 결과 내려받기 (선택)
`stage3_intervention.jsonl`은 사전 등록 기록.


## 9. 행동 실험 v2 — 표집·작동점 게이트 (셀 분리)

greedy는 전 조건 100% camel(천장). v2는 **표집**으로 준수율을 rate화하고, **개입 없이 작동점부터**
찾는다. **셀 A(보정) → 사람이 게이트 확인 → 통과 시에만 셀 B(본 실험).** 본 실험은 게이트 PASS가
기록돼야 실행된다(미통과면 자동 중단). 결과 `stage3_behavior_v2.jsonl`.

In [ ]:
BEH = "/content/drive/MyDrive/instruction-adherence/stage3_behavior_v2.jsonl"
import os; os.makedirs(os.path.dirname(BEH), exist_ok=True)
# ── 셀 A: 작동점 보정만 (개입 없음). 손상 준수율 20~80% 찾기.
# 1) 기본(decision) — formatValue 작업. (이미 천장 확인됨)
# !python src/stage3_behavior.py --calibrate --sample --max-pairs 10 --n-sample-seeds 20 --n-ctx 8 --out "{BEH}"
# 2) stage1 — 결정 작업을 1단계 애매 작업으로 교체(비포화 목적). 지금은 이걸 실행.
!python src/stage3_behavior.py --calibrate --sample --max-pairs 10 --n-sample-seeds 20 --prompt-variant stage1 --out "{BEH}"

### 게이트 확인 (사람)

위 출력의 `→ 게이트 PASS/FAIL` 을 본다. 통과 조건: **손상 20~80% · (준수−손상)≥+20%p · 이름추출≥95%.**
**PASS일 때만** 아래 셀 B를 실행. FAIL이면 n_ctx·작업 지시 등을 **하나씩** 바꿔 보정을 반복.

In [ ]:
# ── 셀 B: 본 L25 개입 (게이트 PASS 기록돼야 실행됨; 아니면 자동 중단).
#   셀 A와 **같은 설정**(n_ctx/temp/top_p/draw)이어야 게이트가 매칭됨.
!python src/stage3_behavior.py --generate --sample --pair-start 10 --n-seeds 3 --n-sample-seeds 20 --n-ctx 8 --out "{BEH}"

In [ ]:
# ── 셀 C: 연쇄(후속 위반) — 아직 greedy. 자기증폭 직접 인과는 별도 2×2 필요(문서 참조).
!python src/stage3_behavior.py --gen-chain --pair-start 10 --n-seeds 3 --n-ctx 8 --out "{BEH}"
!python src/stage3_behavior.py --summary-only --out "{BEH}"

In [ ]:
from google.colab import files
files.download(OUT3)